# MP07 — Synthetic Degradation Laboratory

Expanded engineering lesson; learner mastery unassessed.

## Intuition, definitions and worked mathematics
Let x be H×W×C HR data. A synthetic LR observation is y=D_s(k*x)+n. The Gaussian kernel k has unit sum; s is an integer spatial reduction; n is additive noise in the same numerical units as y. In this laboratory, D_s area-averages nonoverlapping s×s cells. Noise is sampled independently per LR pixel and band from a normal distribution with mean zero and standard deviation tau.

Noise standard deviation has input units; variance tau² has squared units. A random seed selects a reproducible sequence, not an independent physical observation. Noise is not clipped here: Gaussian samples can be negative or exceed the display range. Clipping changes the noise distribution and must be a separate declared choice.

Order matters. Adding independent HR noise with variance tau² before averaging s² pixels reduces its LR variance to tau²/s²; adding noise of variance tau² afterward does not. Blur and noise are simplified modeling choices, not verified sensor physics. The same numerical sigma is measured in HR pixels and may correspond to different physical distances at different ground resolutions.

With a constant image 0.2 and no noise, any normalized blur plus area average should return 0.2. If tau=0.01 is added afterward, expected error mean is zero and expected error MSE is 0.0001 over repeated samples, but one finite sample will differ.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
root=Path.cwd()
if not (root/'src').exists(): root=root.parent
sys.path.insert(0,str(root/'src'))
from rssr.degradation_lab import gaussian,filter_hwc,degrade_hwc,spectrum
x=np.full((64,64,4),.2)
a=degrade_hwc(x,2,.8,.01,17); b=degrade_hwc(x,2,.8,.01,17)
np.testing.assert_array_equal(a,b)
np.testing.assert_allclose(degrade_hwc(x,2,.8,0),.2)
print('sample noise mean/MSE:',float((a-.2).mean()),float(((a-.2)**2).mean()))
assert not np.array_equal(a,degrade_hwc(x,2,.8,.01,18))

## Real-data experiment and interpretation
The real-crop sweep changes scale (2,4), Gaussian width (0,0.8) and LR noise standard deviation (0,0.01), keeping seed 17 and source fixed. Eight area-average conditions plus two unfiltered controls are saved with hashes. They demonstrate parameter effects, not fitted sensor degradation. Do not tune a research model on the held-out scene using these variants.

Run `python scripts/degradation_experiments.py` explicitly from the repository root. The notebook below reads recorded measurements; it does not download data or execute a hidden research run. The source is a checksummed 64×64 NAIP training crop with fixed stored-value/10000 scaling, not verified reflectance.

In [ ]:
p=root/'reports/mp05-mp08-experiments.json'
if p.exists():
    record=json.loads(p.read_text())
    print(record['mp06_mp07_pairs'])
else: print('Real experiment has not run in this checkout.')

## Practice — attempt before reading solutions
1. What LR variance results from averaging independent HR noise of variance 0.0001 at ×2?
2. Why should a noisy result not always equal its expected MSE?
3. What changes if you clip negatives to zero?

## Separate solution guide
1. 0.0001/4=0.000025, assuming four independent samples.
2. An expectation averages hypothetical repeated draws; a finite realization fluctuates.
3. The distribution becomes biased near zero and is no longer the stated additive Gaussian model.

These are generated examples, not evidence that a learner solved the exercises.

In [ ]:
assert np.isclose(.0001/(2**2),.000025)
print('Unclipped minimum example:',degrade_hwc(np.zeros((32,32,1)),2,0,.01).min())

## Reuse, execution and limitations
Use `src/rssr/degradation_lab.py` and `python -m pytest tests/test_degradation_lab.py`. Follow [local and Colab setup](../docs/setup.md); run from notebooks/ or the repository root. The shared NumPy code preserves HWC channels and uses explicit boundaries. No physical sensor calibration or geographic generalization is established.

References: the authoritative stage brief, [dataset card](../docs/dataset-cards.md), [protocol](../docs/protocol.md), and [NumPy FFT documentation](https://numpy.org/doc/stable/reference/routines.fft.html). See [execution evidence](../reports/notebook-execution.json) and [experiment record](../reports/mp05-mp08-experiments.json).